## Preamble

In [ ]:
# Import libraries
import pandas as pd
import plotly.express as px
import plotly.io as pio
import sqlite3

In [ ]:
# Directory
import os
os.chdir('/Users/tasia/Desktop/Data Analysis/Assessment')

In [ ]:
# Load data from database
conn = sqlite3.connect("wdi_cross_sectional_gdp_unemployment_2023.db")

df_gdp = pd.read_sql_query("SELECT * FROM gdp", conn)
df_unp = pd.read_sql_query("SELECT * FROM unemployment", conn)

conn.close()

# Eyeball
print(df_gdp.head())
print(df_unp.head())

In [ ]:
# Convert columns to numeric and replace invalid values with NaN
df_gdp['2023 [YR2023]'] = pd.to_numeric(df_gdp['2023 [YR2023]'], errors='coerce')
df_unp['2023 [YR2023]'] = pd.to_numeric(df_unp['2023 [YR2023]'], errors='coerce')

In [ ]:
# Merge dataframes (inner join!)
df = pd.merge(left = df_gdp, 
              right = df_unp,
              how = 'inner',
              left_on = ['Country Code'],
              right_on = ['Country Code'],
              )
# Rename columns
df = df.rename(columns={
    '2023 [YR2023]_x': 'GDP per capita (2023)',
    '2023 [YR2023]_y': 'Unemployment rate (2023)'})

# Eyeball
df.head()

In [ ]:
# Adding region information
import requests

url = "https://api.worldbank.org/v2/country?format=json&per_page=400"

response = requests.get(url)
response.raise_for_status()
countries = response.json()[1]

# Dataframe with country codes and regions
regions = pd.DataFrame([{"Country Code": country["id"],
                         "Region": country["region"]["value"]}
                        for country in countries])

# Merging region information with GDP and unemployment data
df = pd.merge(df, regions, on="Country Code", how="left")

# Removing missing values and World Bank aggregates
df_regions = df.dropna(subset=["GDP per capita (2023)","Unemployment rate (2023)","Region"])

df_regions = df_regions[df_regions["Region"] != "Aggregates"]

# List available regions
regions_list = sorted(df_regions["Region"].unique())

# Eyeball
df[["Country Name_x", "Region"]].head()

In [ ]:
# Display summary statistics (Table 1)
df.describe()

### Interpretation and Discussing the Output

Looking at summary statistics (also known as descriptive statistics) above, one can see that the mean GDP per capita is 16,838.57 dollars (Constant 2015 US dollars), with a standard deviation of 24,996.31 dollars, indicating a large variation of GDP per capita across countries. In the next column of Table 1 the unemployment rate has a mean value of 6.51 percent, and ranges from a minimum of 0.13 percent to a maximum of 34.61 percent, reflecting substantial variation in unemployment rates across countries.

In Table 1 the median GDP per capita is 6,493.32 dollars which is lower than the mean of 16,838.57 dollars. This suggests a positively skewed distribution. Similar observation of a positively skewed distribution can be seen with the unemployment rate, which for the median is 5.11 percent, a lower percent than the mean of 6.51 percent.

Lastly, the number of observations in Table 1 differs, GDP per capita having 251 “count” and unemployment rate having a 232 “count”. This is due to missing values within the dataset of the unemployment rate.


## Plotting 6 Interactive Graphs

Some graphs and plots include a non-interactive graph/plot versions to show work process

## Bar Charts

In [ ]:
# Plotting a GDP per capita Bar Chart for EU15 Countries in 2023 (Non-interactive)
fig = px.bar(df[(df["Country Name_x"].isin(["Austria",
                                     "Belgium",
                                     "Denmark",
                                     "Finland",
                                     "France",
                                     "Germany",
                                     "Greece",
                                     "Ireland",
                                     "Italy",
                                     "Luxembourg",
                                     "Netherlands",
                                     "Portugal",
                                     "Spain",
                                     "Sweden",
                                     "United Kingdom"]))], 
              x = "Country Name_x", 
              y = "GDP per capita (2023)",
              title = "GDP per capita Across EU15 Countries in 2023")
fig.show()

In [ ]:
# Plotting an Interactive GDP per capita Bar Chart for EU15 Countries in 2023 by European Region (Figure 1)
# Selecting the EU15 countries
eu15 = ["Austria", "Belgium", "Denmark", "Finland", "France", "Germany", "Greece", "Ireland", "Italy", "Luxembourg", "Netherlands", 
        "Portugal", "Spain", "Sweden", "United Kingdom"]

df_eu15 = df[df["Country Name_x"].isin(eu15)].copy()

# Group countries into European regions
regions = {"Northern Europe": ["Denmark", "Finland", "Ireland", "Sweden", "United Kingdom"],
           "Western Europe": ["Austria", "Belgium", "France", "Germany", "Luxembourg", "Netherlands"],
           "Southern Europe": ["Greece", "Italy", "Portugal", "Spain"]}

# Creating a region column
for region, countries in regions.items():
    df_eu15.loc[
        df_eu15["Country Name_x"].isin(countries), "European Region"
    ] = region

# Creating the bar chart
fig = px.bar(
    df_eu15,
    x="Country Name_x",
    y="GDP per capita (2023)",
    title="GDP per capita Across EU15 Countries by European Region in 2023")

# Creating the dropdown menu
buttons = [dict(label="All EU15 countries",method="update",
                args=[{"x": [df_eu15["Country Name_x"].tolist()],
                       "y": [df_eu15["GDP per capita (2023)"].tolist()]}])]

for region in regions:
    region_data = df_eu15[df_eu15["European Region"] == region]

    buttons.append(dict(label=region,method="update",
                        args=[{"x": [region_data["Country Name_x"].tolist()],
                               "y": [region_data["GDP per capita (2023)"].tolist()]}]))

# Adding the dropdown menu above the chart
fig.update_layout(updatemenus=[dict(buttons=buttons,
                                    direction="down",
                                    showactive=True,
                                    x=0,
                                    y=1.35,
                                    xanchor="left",
                                    yanchor="top")],
                  margin=dict(t=180),
                  title=dict(x=0.5,xanchor="center",y=0.90))

fig.show()

### Interpretation and Discussing the Output

Figure 1 above presents an Interactive GDP per capita Bar Chart for EU15 Countries in 2023 by European Region. The horizontal x-axis is the EU15 individual countries, whilst the vertical y-axis in Figure 1 measures the GDP per capita (constant 2015 US dollars) ranging from 0 to 100k dollars (slightly over 100k dollars).

Each of the blue bars in Figure 1 represent the GDP per capita of the corresponding country on the x-axis. The height of the blue bars reflects its respective values on the y-axis.

On the top left corner, underneath the title of Figure 1 a dropdown menu allows for the selection of three European regions. Based on the 15 EU countries chosen the options are; Northern Europe, Western Europe and Southern Europe. The dropdown menu allows for comparison of GDP per capita both across and within European regions.

Choosing the “All EU15 countries” dropdown option in the interactive bar chart of Figure 1 represents the full graphical presentation of the EU15 countries. There’s substantial variation in GDP per capita across the bar chart with Luxembourg having the highest value of 105,422.76 dollars, followed by Ireland at 93,541.96 dollars. In contrast, Greece has the lowest GDP per capita of 21,050.86 dollars, followed by Portugal at 22,417.16 dollars.

When selecting the individual regions through the dropdown menu one can see that countries within Southern Europe have considerably lower GDP per capita relative to those within Northen and Western Europe. Furthermore, there’s relatively limited variation among most Western European countries, except for Luxembourg as it stands out with substantially higher GDP per capita. A similar observation can be made for variation across Northern European countries but with Ireland standing out with higher GDP per capita compared to its’ region.

Overall, Figure 1 highlights the economic disparities that exist both across and within European regions despite the analysis being limited to EU15 countries.

In [ ]:
# Plotting an Unemployment Rate Bar Chart for EU15 Countries in 2023 (Non-interactive)
fig = px.bar(df[(df["Country Name_y"].isin(["Austria",
                                     "Belgium",
                                     "Denmark",
                                     "Finland",
                                     "France",
                                     "Germany",
                                     "Greece",
                                     "Ireland",
                                     "Italy",
                                     "Luxembourg",
                                     "Netherlands",
                                     "Portugal",
                                     "Spain",
                                     "Sweden",
                                     "United Kingdom"]))], 
              x = "Country Name_y", 
              y = "Unemployment rate (2023)",
              title = "Unemployment Rate Across EU15 Countries in 2023")
fig.show()

In [ ]:
# Plotting an Interactive Unemployment Rate Bar Chart for EU15 Countries in 2023 by European Region (Figure 2)
# Selecting EU15 countries
eu15 = ["Austria", "Belgium", "Denmark", "Finland", "France", "Germany", "Greece", "Ireland", "Italy", "Luxembourg", "Netherlands", 
        "Portugal", "Spain", "Sweden", "United Kingdom"]

df_eu15 = df[df["Country Name_y"].isin(eu15)].copy()

# Group countries into European regions
regions = {"Northern Europe": ["Denmark", "Finland", "Ireland", "Sweden", "United Kingdom"],
           "Western Europe": ["Austria", "Belgium", "France", "Germany", "Luxembourg", "Netherlands"],
           "Southern Europe": ["Greece", "Italy", "Portugal", "Spain"]}

# Creating a region column
for region, countries in regions.items():
    df_eu15.loc[df_eu15["Country Name_y"].isin(countries), "European Region"] = region

# Creating the bar chart
fig = px.bar(df_eu15,
    x="Country Name_y",
    y="Unemployment rate (2023)",
    title="Unemployment Rates Across EU15 Countries by European Region in 2023")

# Creating dropdown menu
buttons = [dict(
        label="All EU15 countries",
        method="update",
        args=[{"x": [df_eu15["Country Name_y"].tolist()],
               "y": [df_eu15["Unemployment rate (2023)"].tolist()]}])]

for region in regions:
    region_data = df_eu15[df_eu15["European Region"] == region]

    buttons.append(dict(label=region,
            method="update",
            args=[{"x": [region_data["Country Name_y"].tolist()],
                   "y": [region_data["Unemployment rate (2023)"].tolist()]}]))

# Adding dropdown menu above the chart
fig.update_layout(updatemenus=[dict(buttons=buttons,
                                    direction="down",
                                    showactive=True,
                                    x=0,
                                    y=1.35,
                                    xanchor="left",
                                    yanchor="top")],
                  margin=dict(t=180),
                  title=dict(
                      x=0.5,
                      xanchor="center",
                      y=0.90),yaxis_title="Unemployment rate (%)")
fig.show()

### Interpretation and Discussing the Output

Figure 2 above presents an Interactive Unemployment Rate Bar Chart for EU15 Countries in 2023 by European Region. Similar to Figure 1 the horizontal x-axis represents the individual EU15 countries, however the vertical y-axis in Figure 2 measures the unemployment rate as a percentage. The height of each blue bar reflects the unemployment rate of the corresponding country.

The dropdown menu in the top left corner, underneath the title of Figure 2 follows the same structure as Figure 1. Once again, the dropdown menu allows for comparison of unemployment rates both across and within European regions.

Choosing the “All EU15 countries” dropdown option in the interactive bar chart of Figure 2 represents the full graphical presentation of unemployment rates across the EU15 countries. There is substantial variation in unemployment rates across the bar chart with Spain having the highest unemployment rate of 12.18 percent, followed by Greece at 11.02 percent. In contrast, Germany has the lowest unemployment rate of 3.07 percent, followed by the Netherlands at 3.54 percent.

When selecting the individual regions through the dropdown menu one can see that the countries within Southern Europe generally have higher unemployment rates relative to those within Northern and Western Europe. Within Western Europe, France stands out with a relatively high unemployment rate of 7.34 percent. Whereas in Northern Europe, Sweden and Finland have unemployment rates of 7.61 percent and 7.15 percent.

Interestingly when comparing Figure 1 to Figure 2 one can see that Southern European countries generally have both lower GDP per capita and higher unemployment rates relative to the other two regions.

Overall, Figure 2 highlights the disparities in labor market conditions that exist both across and within European regions.

In [ ]:
# Plotting a Clustered Bar Chart of All Countries by World Bank Region in 2023 (Non-interactive)

# Calculating average GDP per capita
average_gdp = df_regions["GDP per capita (2023)"].mean()

# Grouping countries by GDP per capita
df_clustered = df_regions.copy()

df_clustered["GDP Group"] = "Below-average GDP"
df_clustered.loc[df_clustered["GDP per capita (2023)"] >= average_gdp, "GDP Group"] = "Above-average GDP"

# Calculating average unemployment by region and GDP group
df_clustered = df_clustered.groupby(["Region", "GDP Group"], as_index = False)["Unemployment rate (2023)"].mean()

# Plotting clustered bar chart
fig = px.bar(df_clustered,
             x = "Region",
             y = "Unemployment rate (2023)",
             color = "GDP Group",
             barmode = "group",
             title = "Average Unemployment by World Bank Region and GDP group in 2023")

fig.update_layout(yaxis_title = "Average unemployment rate (%)")

fig.show()

In [ ]:
# Plotting an Interactive Clustered Bar Chart of All Countries by World Bank Region in 2023 (Figure 3)
fig = px.bar(df_clustered,
             x = "Region",
             y = "Unemployment rate (2023)",
             color = "GDP Group",
             barmode = "group",
             title = "Average Unemployment by World Bank Region and GDP group in 2023")

# Creating dropdown buttons
regions_list = sorted(df_clustered["Region"].unique())

buttons = [dict(label = "All regions",
                method = "update",
                args = [{"x": [df_clustered[df_clustered["GDP Group"] == trace.name]["Region"].tolist() for trace in fig.data],
                         "y": [df_clustered[df_clustered["GDP Group"] == trace.name]["Unemployment rate (2023)"].tolist() for trace in fig.data]}])]

for region in regions_list:
    region_data = df_clustered[df_clustered["Region"] == region]

    buttons.append(dict(label = region,
                        method = "update",
                        args = [{"x": [region_data[region_data["GDP Group"] == trace.name]["Region"].tolist() for trace in fig.data],
                                 "y": [region_data[region_data["GDP Group"] == trace.name]["Unemployment rate (2023)"].tolist() for trace in fig.data]}]))

# Adding dropdown menu
fig.update_layout(updatemenus = [dict(buttons = buttons,
                        direction = "down",
                        showactive = True,
                        x = 0,
                        y = 1.35,
                        xanchor = "left",
                        yanchor = "top")],
    margin = dict(t = 180),
    title = dict(x = 0.5,
                 xanchor = "center",
                 y = 0.90),
    yaxis_title = "Average unemployment rate (%)")

fig.show()

### Interpretation and Discussing the Output

Figure 3 above presents an Interactive Clustered Bar Chart of Average Unemployment by World Bank Region and GDP Group in 2023. Unlike Figures 1 and 2, the horizontal x-axis represents the World Bank regions rather than individual countries, whilst the vertical y-axis measures the average unemployment rate as a percentage.

The blue bars in Figure 3 represent the average unemployment rate of countries with GDP per capita equal to or above the sample mean. The red bars represent those with GDP per capita below the sample mean. Similarly to the previous figures, the dropdown menu allows for comparison both across and within regions.

Choosing the “All regions” dropdown option represents the full graphical presentation of the seven World Bank regions. The largest difference between the two GDP groups can be seen within the Middle East, North Africa, Afghanistan and Pakistan region. This is where countries with below average GDP per capita have an average unemployment rate of 13.12 percent, compared to 2.43 percent for countries with above-average GDP per capita. A similar pattern, although considerably smaller in magnitude, can be seen within Europe and Central Asia, with average unemployment rates of 7.20 percent and 5.62 percent, respectively.

However, the opposite pattern can be observed within East Asia and Pacific and Latin America and Caribbean, in which countries with above average GDP per capita have higher average unemployment rates relative to those with below average GDP per capita.

Furthermore, when selecting the individual regions through the dropdown menu, one can see that North America only has a blue bar, whilst South Asia and Sub-Saharan Africa only have red bars. This is due to the countries within these regions belonging to only one of the two GDP groups in the sample.

Overall, Figure 3 highlights that higher GDP per capita is not consistently associated with lower unemployment rates across all regions. This suggests that the relationship between the two variables may differ depending on the region.


## Scatter Plots

In [ ]:
# Plotting a Scatter Plot of GDP per capita and Unemployment in 2023 (Non-interactive)
fig = px.scatter(df, 
                 x = "GDP per capita (2023)", 
                 y = "Unemployment rate (2023)",
                 hover_data = ["Country Name_x"],
                 title = "GDP per capita and Unemployment Rates in 2023")
fig.show()

In [ ]:
# Plotting an Interactive Scatter Plot of GDP per capita and Unemployment in 2023 by Region (Figure 4)
# Creating the interactive scatter plot with region selection
fig = px.scatter(
    df_regions,
    x="GDP per capita (2023)",
    y="Unemployment rate (2023)",
    color="Region",
    hover_name="Country Name_x",
    title="GDP per capita and Unemployment by Region in 2023")

# Creating dropdown buttons for each region
regions_list = sorted(df_regions["Region"].unique())

buttons = [dict(
        label="All regions",
        method="update",
        args=[{"visible": [True] * len(fig.data)}])]

for region in regions_list:
    buttons.append(dict(label=region,
            method="update",
            args=[{"visible": [trace.name == region for trace in fig.data]}]))


# Adding dropdown menu above the scatter plot
fig.update_layout(updatemenus=[dict(
    buttons=buttons,
    direction="down",
    showactive=True,
    x=0,
    y=1.35,
    xanchor="left",
    yanchor="top")],
                  margin=dict(t=180),
                  title=dict(
                      x=0.5,
                      xanchor="center",
                      y=0.90))
fig.show()

### Interpretation and Discussing the Output

Figure 4 above presents an Interactive Scatter Plot of GDP per capita and Unemployment by Region in 2023. The horizontal x-axis measures GDP per capita (constant 2015 US dollars) ranging from 0 to slightly over 100k dollars, whilst the vertical y-axis measures the unemployment rate as a percentage ranging from 0 to slightly over 30 percent.

Each dot in Figure 4 represents an individual country observation. The different colors represent the seven World Bank regions. Similarly to Figure 3, the dropdown menu allows for the selection of individual regions. However, in Figure 4 hovering over each dot allows one to identify the corresponding country and its respective GDP per capita and unemployment rate.

Choosing the “All regions” dropdown option represents the full graphical presentation of the scatter plot. There is substantial dispersion in unemployment rates among countries with lower GDP per capita, whilst countries with higher GDP per capita generally have lower and less dispersed unemployment rates. However, the overall distribution of the observations does not appear to follow a strong linear pattern.

Looking at two regional examples, first being Europe and Central Asia. When selecting Europe and Central Asia through the dropdown menu one can observe that countries with higher GDP per capita generally have lower unemployment rates. However, there are exceptions to this pattern, such as Moldova, which has a relatively low GDP per capita of 3,758.37 dollars alongside an unemployment rate of 1.57 percent.

In contrast, the second regional example being Sub-Saharan Africa, when selecting it a different pattern can be observed, with several countries with relatively higher GDP per capita having higher unemployment rates within the region. For instance, Eswatini has a GDP per capita of 3,790.17 dollars alongside an unemployment rate of 34.61 percent, whilst Niger has a GDP per capita of 561.96 dollars and an unemployment rate of only 0.41 percent.

Overall, Figure 4 further supports the observations made in Figure 3, suggesting that the relationship between GDP per capita and unemployment differs across regions. Whilst a negative association can be observed within certain regions, higher GDP per capita does not necessarily correspond to lower unemployment rates across all countries.

## Histograms

In [ ]:
# Plotting a Bivariate Histogram (Heatmap) of GDP per capita and Unemployment in 2023 (Non-interactive)
fig = px.density_heatmap(df, 
                         x = "GDP per capita (2023)", 
                         y = "Unemployment rate (2023)",
                         nbinsx = 30, 
                         nbinsy = 30)
fig.show()

In [ ]:
# Plotting an Interactive Bivariate Histogram (Heatmap) of GDP per capita and Unemployment in 2023 by Region (Figure 5)
# Create a heatmap for all regions
fig = px.density_heatmap(df_regions,
                         x="GDP per capita (2023)",
                         y="Unemployment rate (2023)",
                         nbinsx=30,
                         nbinsy=30,
                         title="GDP per capita and Unemployment by Region in 2023")

# Creating dropdown buttons
buttons = [dict(label="All regions",method="update",args=[{"x": [df_regions["GDP per capita (2023)"].tolist()],
                                                           "y": [df_regions["Unemployment rate (2023)"].tolist()]}])]

for region in regions_list:
    region_data = df_regions[df_regions["Region"] == region]

    buttons.append(dict(label=region,method="update",args=[{"x": [region_data["GDP per capita (2023)"].tolist()],
                                                            "y": [region_data["Unemployment rate (2023)"].tolist()]}]))
    
# Adding dropdown menu above the heatmap
fig.update_layout(updatemenus=[dict(buttons=buttons,
                                    direction="down",
                                    showactive=True,
                                    x=0,
                                    y=1.35,
                                    xanchor="left",
                                    yanchor="top")],
                  margin=dict(t=180),
                  title=dict(
                      x=0.5,
                      xanchor="center",
                      y=0.90))

fig.show()

### Interpretation and Discussing the Output

Figure 5 above presents an Interactive Heatmap of GDP per capita and Unemployment by Region in 2023. Similar to Figure 4 the horizontal x-axis measures GDP per capita (constant 2015 US dollars) ranging from 0 to slightly over 100k dollars, whilst the vertical y-axis measures the unemployment rate as a percentage ranging from 0 to 35 percent.

Unlike the individual dots in Figure 4, the colored rectangles in Figure 5 represent the number of country observations falling within particular ranges of GDP per capita and unemployment. The color scale on the right-hand side of Figure 5 measures the count of observations, with dark blue representing no observations and colors progressing towards yellow representing a higher concentration of observations.

The dropdown menu follows the same structure as Figure 4. It allows for comparison of the distribution of observations across the seven World Bank regions.

When selecting Europe and Central Asia through the dropdown menu (looking at the same regional examples as Figure 4), one can observe that the colored rectangles are dispersed across a relatively wide range of GDP per capita values. The maximum count of observations within an individual rectangle is two. Furthermore, countries with GDP per capita exceeding 40,000 dollars are predominantly concentrated within unemployment rates of approximately 3 to 8 percent.

In contrast when selecting Sub-Saharan Africa, there is a considerably greater concentration of observations towards the bottom left corner of the heatmap. The yellow rectangle represents a count of six observations with GDP per capita ranging from approximately 500 to 1,000 dollars and unemployment rates ranging from 2 to 4 percent. However, there are also several dispersed observations with unemployment rates exceeding 20 percent.

It is important to acknowledge that the color scale and axis ranges adjust when selecting individual regions through the dropdown menu. Thus, the same color does not necessarily represent the same number of observations across different regions.

Overall, Figure 5 complements the scatter plot in Figure 4 by highlighting the concentration and dispersion of observations. This further demonstrates differences in the distribution of GDP per capita and unemployment rates across regions.

In [ ]:
# Plotting a Heatmap with Marginal Histograms of GDP per capita and Unemployment in 2023 (Non-interactive)
fig = px.density_heatmap(df, 
                         x = "GDP per capita (2023)", 
                         y = "Unemployment rate (2023)",
                         marginal_x = "histogram", 
                         marginal_y = "histogram",
                         nbinsx = 30, 
                         nbinsy = 30)
fig.show()

In [ ]:
# Plotting an Interactive Heatmap with Marginal Histograms of GDP per capita and Unemployment in 2023 by Region (Figure 6)
# Using the same valid country observations for every selection
df_heatmap = df_regions.copy()

# Creating the interactive heatmap
fig = px.density_heatmap(df_heatmap,
                         x="GDP per capita (2023)",
                         y="Unemployment rate (2023)",
                         marginal_x="histogram",
                         marginal_y="histogram",
                         nbinsx=30,
                         nbinsy=30,
                         title="GDP per capita and Unemployment with Marginal Histograms by Region in 2023")

# Creating dropdown options
regions_list = sorted(df_heatmap["Region"].dropna().unique())

# Heatmap, top histogram and right histogram

def make_region_update(data):
    gdp = data["GDP per capita (2023)"].tolist()
    unemployment = data["Unemployment rate (2023)"].tolist()

    return {"x": [gdp, gdp, None], "y": [unemployment, None, unemployment]}

buttons = [dict(label="All regions", method="update", args=[make_region_update(df_heatmap)])]

for region in regions_list:
    region_data = df_heatmap[df_heatmap["Region"] == region]
    
    buttons.append(dict(label=region,
                        method="update",
                        args=[make_region_update(region_data)]))

# Adding dropdown above the chart
fig.update_layout(updatemenus=[dict(buttons=buttons,
                                    direction="down",
                                    showactive=True,
                                    x=0,
                                    y=1.35,
                                    xanchor="left",
                                    yanchor="top")],
                  margin=dict(t=180),
                  title=dict(x=0.5,
                      xanchor="center",
                             y=0.90))

fig.show()

### Interpretation and Discussing the Output

Figure 6 above presents an Interactive Heatmap of GDP per capita and Unemployment with Marginal Histograms by Region in 2023. The horizontal x-axis and vertical y-axis, alongside the color scale and dropdown menu follow the same structure as Figure 5.

However, unlike Figure 5, Figure 6 includes two additional histograms. The histogram above the heatmap represents the frequency distribution of GDP per capita, whilst the histogram on the right-hand side represents the frequency distribution of unemployment rates. The height of the bars in the upper histogram and the length of the bars in the right-hand histogram reflect the respective number of country observations falling within each interval.

When selecting Europe and Central Asia through the dropdown menu (following the same two regional examples for consistency), one can observe that the upper histogram presents a positively skewed distribution, with a greater concentration of observations towards the lower end of GDP per capita and fewer observations towards the higher end. Looking at the histogram on the right-hand side there is a greater concentration of observations within unemployment rates of approximately 3 to 7 percent, with relatively fewer observations exceeding 10 percent.

Similarly, when selecting Sub-Saharan Africa, one can observe a positively skewed distribution of GDP per capita, with a substantial concentration of observations below 2,000 dollars. Furthermore, the histogram on the right-hand side also presents a positively skewed distribution, with a greater concentration of observations towards lower unemployment rates, despite the presence of extreme values exceeding 30 percent.

The positively skewed distributions observed in Figure 6 are consistent with the observations made earlier when discussing the descriptive statistics in Table 1. In Table 1 the mean values of both variables exceeded their respective median values.

Overall, Figure 6 extends the graphical presentation of Figure 5 by allowing one to observe both the joint distribution of GDP per capita and unemployment and the individual distribution of each variable across regions.

## Bubble Chart

In [ ]:
# Downloading Population Data for 2023
import requests

url = "https://api.worldbank.org/v2/country/all/indicator/SP.POP.TOTL?format=json&date=2023&per_page=500"
data = requests.get(url).json()[1]

# Creating dataframe
df_population = pd.DataFrame(data)
df_population["Country Code"] = df_population["countryiso3code"]
df_population["Population (2023)"] = df_population["value"]

# Merge dataframes (inner join!)
df_bubble = pd.merge(left = df_regions,
                     right = df_population[["Country Code", "Population (2023)"]],
                     how = "inner",
                     on = "Country Code")

# Remove missing population values
df_bubble = df_bubble.dropna(subset = ["Population (2023)"])

# Eyeball
df_bubble.head()

In [ ]:
# Plotting an Interactive Bubble Chart of GDP per capita and Unemployment in 2023 by Region (Figure 7)
fig = px.scatter(df_bubble,
                 x = "GDP per capita (2023)",
                 y = "Unemployment rate (2023)",
                 size = "Population (2023)",
                 color = "Region",
                 hover_name = "Country Name_x",
                 size_max = 55,
                 title = "GDP per capita, Unemployment and Population by Region in 2023")

# Creating dropdown buttons for each region
regions_list = sorted(df_bubble["Region"].unique())

buttons = [dict(label = "All regions",
                method = "update",
                args = [{"visible": [True] * len(fig.data)}])]

for region in regions_list:
    buttons.append(dict(label = region,
                        method = "update",
                        args = [{"visible": [trace.name == region for trace in fig.data]}]))

# Adding dropdown menu
fig.update_layout(updatemenus = [dict(buttons = buttons,
                        direction = "down",
                        showactive = True,
                        x = 0,
                        y = 1.35,
                        xanchor = "left",
                        yanchor = "top")],
    margin = dict(t = 180),
    title = dict(x = 0.5,
                 xanchor = "center",
                 y = 0.90))

fig.show()

### Interpretation and Discussing the Output

Figure 7 above presents an Interactive Bubble Chart of GDP per capita, Unemployment and Population by Region in 2023. Similarly to Figure 4 the horizontal x-axis measures GDP per capita (constant 2015 US dollars). Whereas the vertical y-axis measures the unemployment rate as a percentage.

Each bubble in Figure 7 represents an individual country observation, with the different colors representing the respective World Bank regions. However, unlike Figure 4 the size of each bubble represents the total population of the corresponding country. Meaning the larger the bubble, the larger the population of the country. The dropdown menu follows the same structure as the previous figures up to figure 3.

For this Figure one will look at two new regional examples. First when selecting South Asia through the dropdown menu, one can observe that India stands out with the largest bubble, having a GDP per capita of approximately 2,200 dollars and an unemployment rate of 4.2 percent. Despite India having a considerably larger population relative to the other countries within its region, it does not have the highest GDP per capita nor the highest unemployment rate.

When selecting East Asia and Pacific, China stands out with the largest bubble, having a GDP per capita of approximately 12,500 dollars and an unemployment rate of 4.7 percent. Furthermore, there is substantial variation in GDP per capita across countries within this region, with several smaller bubbles like Australia and Singapore appearing further towards the right-hand side of the horizontal x-axis.

When comparing the two regions, one can observe that whilst India and China have similarly large populations and relatively similar unemployment rates, there is a substantial difference between their respective GDP per capita values. Meaning, that countries with similar population sizes do not necessarily have similar economic conditions.

Overall, Figure 7 extends the graphical presentation of Figure 4 through the introduction of population as an additional variable. This provides a broader understanding of the economic and demographic disparities that exist both across and within World Bank regions.